# Annexe B — Cahier de code, Chapitre 15
## Fonctions de coût (deep learning)

Ce notebook accompagne le chapitre 15 de *Fondamentaux de la Vision par
Ordinateur*. Pour **chaque sous-chapitre**, une cellule de code Python montre la
**syntaxe et l'usage** de la notion — au plus simple, sans fonctions ni gestion
d'erreurs. Le but n'est pas de produire du code de production, mais de relier la
formule du livre à son équivalent Python.

> **Pré-requis** : `pip install numpy scipy scikit-image scikit-learn opencv-python matplotlib`

Exécutez les cellules dans l'ordre : la première prépare les données d'entrée.

## Préparation des données

In [ ]:
# Tout en NumPy pour montrer la formule ; pas de framework
import numpy as np

logits = np.array([2.0, 0.5, -1.0])     # scores bruts d'un classifieur
cible = 0                               # classe correcte

## 15.1 — Entropie croisée et softmax

In [ ]:
exp = np.exp(logits - logits.max())
proba = exp / exp.sum()                 # softmax
perte = -np.log(proba[cible])           # cross-entropy
print(proba, perte)

## 15.2 — Dice loss

In [ ]:
# 1 − Dice, sur des cartes de proba (segmentation)
pred = np.array([0.2, 0.8, 0.9, 0.1])
gt   = np.array([0.0, 1.0, 1.0, 0.0])
dice = 2 * (pred * gt).sum() / (pred.sum() + gt.sum())
print(1 - dice)

## 15.3 — Focal loss

In [ ]:
# pondère les exemples faciles vers le bas (gamma)
p = 0.9                                 # proba donnée à la bonne classe
gamma = 2.0
focal = -(1 - p) ** gamma * np.log(p)
print(focal)

## 15.4 — Smooth L1 (Huber)

In [ ]:
# quadratique près de 0, linéaire au-delà (robuste aux aberrations)
err = np.array([0.3, 2.5, -1.2])
beta = 1.0
sl1 = np.where(np.abs(err) < beta, 0.5 * err ** 2 / beta, np.abs(err) - 0.5 * beta)
print(sl1)

## 15.5 — IoU loss et GIoU

In [ ]:
# sur deux boîtes [x1, y1, x2, y2]
A = np.array([0, 0, 2, 2]); B = np.array([1, 1, 3, 3])
xi = max(A[0], B[0]); yi = max(A[1], B[1])
xa = min(A[2], B[2]); ya = min(A[3], B[3])
inter = max(0, xa - xi) * max(0, ya - yi)
union = 4 + 4 - inter
iou = inter / union
print("IoU loss :", 1 - iou)

## 15.6 — Loss contrastive (InfoNCE)

In [ ]:
# rapproche une paire positive, éloigne les négatives (température tau)
sim = np.array([0.9, 0.2, 0.1, 0.3])    # similarité ancre↔{positif, négatifs...}
tau = 0.1
e = np.exp(sim / tau)
infonce = -np.log(e[0] / e.sum())
print(infonce)